analyse feature importance, correlation, and what you have decided to engineer and why.

In [41]:
import pandas as pd
from pathlib import Path

df = pd.read_csv(Path.cwd().parent.parent / "archive" / "integrated_data.csv")

df.head()

,id,date,client_id,card_id,amount,use_chip,errors,current_age,retirement_age,birth_year,...,card_brand,card_type,expires,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web,target
0,7475327,2010-01-01 00:01:00,1556,2972,-77.00,Swipe Transaction,NaN,30,67,1989,...,Mastercard,Debit (Prepaid),2022-07-01,1,2,55.0,2008-05-01,2008-01-01,0,0.0
1,7475328,2010-01-01 00:02:00,561,4575,14.57,Swipe Transaction,NaN,48,67,1971,...,Mastercard,Credit,2024-12-01,1,1,9100.0,2005-09-01,2015-01-01,0,0.0
2,7475329,2010-01-01 00:02:00,1129,102,80.00,Swipe Transaction,NaN,49,65,1970,...,Mastercard,Debit,2020-05-01,1,1,14802.0,2006-01-01,2008-01-01,0,0.0
3,7475331,2010-01-01 00:05:00,430,2860,200.00,Swipe Transaction,NaN,52,67,1967,...,Mastercard,Debit,2024-10-01,0,2,37634.0,2004-05-01,2006-01-01,0,NaN
4,7475332,2010-01-01 00:06:00,848,3915,46.41,Swipe Transaction,NaN,51,69,1968,...,Visa,Debit,2020-01-01,1,1,19113.0,2009-07-01,2014-01-01,0,0.0


In [42]:
df.columns

Index(['id', 'date', 'client_id', 'card_id', 'amount', 'use_chip', 'errors',
       'current_age', 'retirement_age', 'birth_year', 'birth_month', 'gender',
       'per_capita_income', 'yearly_income', 'total_debt', 'credit_score',
       'num_credit_cards', 'card_brand', 'card_type', 'expires', 'has_chip',
       'num_cards_issued', 'credit_limit', 'acct_open_date',
       'year_pin_last_changed', 'card_on_dark_web', 'target'],
      dtype='object')

1. Financial Features

In [43]:
# debt to income ratio
df['debt_to_income_ratio'] = df['total_debt'] / df['yearly_income']

2. Age-based Features

In [54]:
# years to retirement as of current age
df["years_to_retirement"] = (df["retirement_age"] - df["current_age"]).clip(lower=0)

# flag retirement status
df['is_retired'] = (df['current_age'] >= df['retirement_age']).to_numpy().astype(int)

3. Card Lifecycle Features

In [50]:
import numpy as np

# important now since we load from csv, DO NOT USE IN PIPELINE!!!
df["date"] = pd.to_datetime(df["date"])
df["acct_open_date"] = pd.to_datetime(df["acct_open_date"])
df["expires"] = pd.to_datetime(df["expires"])

# card age in months
df["card_age_months"] = (
    (df["date"] - df["acct_open_date"])
).dt.days // 30

''' since pin changes are also recorded after the date of transaction this is left out for now.
# pin change age in months
df["days_since_pin_change"] = (
    df["date"] - df["year_pin_last_changed"]
).dt.days
'''

# time until card expires, older cards could be more prone to fraud
df["days_until_expiry"] = (
    df["expires"] - df["date"]
).dt.days

In [51]:
df["hour"] = df["date"].dt.hour
df["dayofweek"] = df["date"].dt.dayofweek
df["is_night"] = df["hour"].between(0, 5).astype(int)

In [52]:
df.columns

Index(['id', 'date', 'client_id', 'card_id', 'amount', 'use_chip', 'errors',
       'current_age', 'retirement_age', 'birth_year', 'birth_month', 'gender',
       'per_capita_income', 'yearly_income', 'total_debt', 'credit_score',
       'num_credit_cards', 'card_brand', 'card_type', 'expires', 'has_chip',
       'num_cards_issued', 'credit_limit', 'acct_open_date',
       'year_pin_last_changed', 'card_on_dark_web', 'target',
       'debt_to_income_ratio', 'years_to_retirement', 'is_retired',
       'card_age_months', 'hour', 'dayofweek', 'is_night',
       'days_until_expiry'],
      dtype='object')

In [55]:
df.head()

,id,date,client_id,card_id,amount,use_chip,errors,current_age,retirement_age,birth_year,...,card_on_dark_web,target,debt_to_income_ratio,years_to_retirement,is_retired,card_age_months,hour,dayofweek,is_night,days_until_expiry
0,7475327,2010-01-01 00:01:00,1556,2972,-77.00,Swipe Transaction,NaN,30,67,1989,...,0,0.0,2.281687,37,0,20,0,4,1,4563
1,7475328,2010-01-01 00:02:00,561,4575,14.57,Swipe Transaction,NaN,48,67,1971,...,0,0.0,3.042873,19,0,52,0,4,1,5447
2,7475329,2010-01-01 00:02:00,1129,102,80.00,Swipe Transaction,NaN,49,65,1970,...,0,0.0,1.060698,16,0,48,0,4,1,3772
3,7475331,2010-01-01 00:05:00,430,2860,200.00,Swipe Transaction,NaN,52,67,1967,...,0,NaN,2.411921,15,0,69,0,4,1,5386
4,7475332,2010-01-01 00:06:00,848,3915,46.41,Swipe Transaction,NaN,51,69,1968,...,0,0.0,1.406951,18,0,6,0,4,1,3651


In [56]:
df.drop(columns=['id', 'date', 'client_id', 'card_id', 'current_age', 'retirement_age', 'per_capita_income', 'acct_open_date',
       'year_pin_last_changed'], inplace=True)

In [57]:
df.head()

,amount,use_chip,errors,birth_year,birth_month,gender,yearly_income,total_debt,credit_score,num_credit_cards,...,card_on_dark_web,target,debt_to_income_ratio,years_to_retirement,is_retired,card_age_months,hour,dayofweek,is_night,days_until_expiry
0,-77.00,Swipe Transaction,NaN,1989,7,Female,48277.0,110153.0,740,4,...,0,0.0,2.281687,37,0,20,0,4,1,4563
1,14.57,Swipe Transaction,NaN,1971,6,Male,36853.0,112139.0,834,5,...,0,0.0,3.042873,19,0,52,0,4,1,5447
2,80.00,Swipe Transaction,NaN,1970,4,Male,34449.0,36540.0,686,3,...,0,0.0,1.060698,16,0,48,0,4,1,3772
3,200.00,Swipe Transaction,NaN,1967,5,Female,53350.0,128676.0,685,5,...,0,NaN,2.411921,15,0,69,0,4,1,5386
4,46.41,Swipe Transaction,NaN,1968,5,Male,68362.0,96182.0,711,2,...,0,0.0,1.406951,18,0,6,0,4,1,3651


In [60]:
train = df[df["target"].notna()]
print(f"Training set size: {len(train)} ({len(train)/len(df)*100:.2f}%)")

test = df[df["target"].isna()]
print(f"Test set size: {len(test)} ({len(test)/len(df)*100:.2f}%)")

Training set size: 8914963 (67.00%)
Test set size: 4390952 (33.00%)
